This notebook will utilize the GSTools and PyKrige libraries to implement Ordinary Kriging and Universal Kriging for point cloud data. Data will be interpolated onto a grid with horizontal spatial resolution of your chosing. This grid may be clipped by a boundary shapefile if you have one.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys
sys.path.insert(0, os.path.abspath('src'))
import geopandas as gpd
import pandas as pd
from shapely import wkt
import gstools as gs
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt

import OK, UK, RK

In [ ]:
boundary_gdf = gpd.read_file(f'{os.path.abspath('raw_data')}/boundary.shp')
pts_df = pd.read_csv(f'{os.path.abspath("raw_data")}/sampled_pts.csv')
pts_df['geometry'] = pts_df['geometry'].apply(wkt.loads)
example_pts = gpd.GeoDataFrame(pts_df, geometry='geometry', crs=boundary_gdf.crs)

In [ ]:
# First, extract the x, y, z data from your dataset

# this is for my example data in the raw_data directory

x = example_pts.xLocation.values
y = example_pts.yLocation.values
z = example_pts.Z_use.values

In [ ]:
# next establish what variogram models you wish to test with GSTools

# some typical ones I default to the below, since some GSTools models are not implemented within PyKrige

vario_models = {
    "Gaussian": gs.Gaussian,
    "Exponential": gs.Exponential,
    "Matern": gs.Matern,
    "Stable": gs.Stable,
    "Rational": gs.Rational,
    "Circular": gs.Circular,
    "Spherical": gs.Spherical,
    "SuperSpherical": gs.SuperSpherical,
    "JBessel": gs.JBessel,
}

# establish the estimator used to auto-fit the variogram and determine its hyperparameters
estimator = 'cressie'   # or 'matheron'. Two different approaches to weighted least square solution for automatic variogram fitting. Cressie is more robust to outliers

# establish the horizontal spatial resolution of your outputs
pred_grid_resolution = 10.0     # this will be in the unit of your CRS. For the example data in raw_data this is feet. You can adjust to be the size you wish, preferably a whole number

# Kriging

In [ ]:
# you can change these to save whereever you like
ok_pred_out_path = f'{os.path.abspath('model_outputs')}/OK_test_predictions.tif'      
ok_var_out_path = f'{os.path.abspath('model_outputs')}/OK_test_variances.tif'

ok = OK.OK_AIC(
    vario_models, x, y, z, boundary_gdf, 
    ok_pred_out_path, ok_var_out_path, 
    vario_estimator = estimator, 
    grid_res = pred_grid_resolution, 
    plot_outputs = True
    )

pred_path, var_path = ok.generate()

In [ ]:
uk_trend = "regional_linear" # or None. There are more options, but I have not explored or developed those within the UK_AIC class yet

uk = UK.UK_AIC(
    vario_models, x, y, z, boundary_gdf,
    'model_outputs/UK_test_predictions.tif', 'model_outputs/UK_test_variances.tif', 
    trend_model = uk_trend,
    vario_estimator = estimator,
    grid_res=10.0,
    plot_outputs=True,
)

pred_path, var_path = uk.generate()

In [ ]:
# I employed something like this for my presentation at hydroreads. Though instead of sklearn regression models, I was using Verde Green's function tension splines

rk = RK.RK_AIC(
    x, y, z, boundary_gdf,
    pred_out_path="model_outputs/RK_test_predictions.tif",
    var_out_path="model_outputs/RK_test_variances.tif",
    regression_model="svr",
    regression_model_params=(),         # you can head to sklearn to look at regression models and parameters for input as a dictionary. e.g., {'random_state': 42, 'n_estiamtors': 300}
    vario_models=vario_models,          # same dictionary you use for OK/UK
    vario_estimator=estimator,          # same as OK/UK
    grid_res=pred_grid_resolution,
    plot_outputs=True,
)
pred_path, var_path = rk.generate()

# Leave one out cross validation
- particularly useful when you have a small number of sampled points
- If you have anything past ~50 points, spatial block CV is recommended, though it is not implemented in this repository

In [ ]:
# LOOCV for Ordinary Kriging

ok_cv_results = OK.ok_loocv(x, y, z, boundary_gdf, vario_models, 
                          vario_estimator=estimator, grid_res=pred_grid_resolution)

print("Ordinary Kriging LOOCV Results:")
print(f"  RMSE: {ok_cv_results['rmse']:.4f}")
print(f"  MAE:  {ok_cv_results['mae']:.4f}")
print(f"  R²:   {ok_cv_results['r2']:.4f}")

fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
ax.scatter(ok_cv_results['actuals'], ok_cv_results['predictions'], alpha=0.6)
ax.plot([z.min(), z.max()], [z.min(), z.max()], 'r--', lw=2, label='Perfect')
ax.set_xlabel('Actual Values')
ax.set_ylabel('LOOCV Predictions')
ax.set_title('OK LOOCV: Actual vs Predicted')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# LOOCV for Universal Kriging
uk_trend = "regional_linear" # or None. There are more options, but I have not explored or developed those within the UK_AIC class yet

uk_cv_results = UK.uk_loocv(x, y, z, boundary_gdf, vario_models, 
                         trend_model=uk_trend, vario_estimator=estimator, 
                         grid_res=pred_grid_resolution)

print("Universal Kriging LOOCV Results:")
print(f"  RMSE: {uk_cv_results['rmse']:.4f}")
print(f"  MAE:  {uk_cv_results['mae']:.4f}")
print(f"  R²:   {uk_cv_results['r2']:.4f}")

fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
ax.scatter(uk_cv_results['actuals'], uk_cv_results['predictions'], alpha=0.6, color='orange')
ax.plot([z.min(), z.max()], [z.min(), z.max()], 'r--', lw=2, label='Perfect')
ax.set_xlabel('Actual Values')
ax.set_ylabel('LOOCV Predictions')
ax.set_title('UK LOOCV: Actual vs Predicted')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# LOOCV for Regression Kriging

rk_cv_results = RK.rk_loocv(x, y, z, boundary_gdf, regression_model='svr', 
                         vario_models=vario_models, vario_estimator=estimator,
                         grid_res=pred_grid_resolution)

print("Regression Kriging LOOCV Results:")
print(f"  RMSE: {rk_cv_results['rmse']:.4f}")
print(f"  MAE:  {rk_cv_results['mae']:.4f}")
print(f"  R²:   {rk_cv_results['r2']:.4f}")

fig, ax = plt.subplots(figsize=(8, 8), dpi=150)
ax.scatter(rk_cv_results['actuals'], rk_cv_results['predictions'], alpha=0.6, color='green')
ax.plot([z.min(), z.max()], [z.min(), z.max()], 'r--', lw=2, label='Perfect')
ax.set_xlabel('Actual Values')
ax.set_ylabel('LOOCV Predictions')
ax.set_title('RK LOOCV: Actual vs Predicted')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Compare all three methods side-by-side

comparison = pd.DataFrame({
    'Method': ['OK', 'UK', 'RK'],
    'RMSE': [ok_cv_results['rmse'], uk_cv_results['rmse'], rk_cv_results['rmse']],
    'MAE': [ok_cv_results['mae'], uk_cv_results['mae'], rk_cv_results['mae']],
    'R²': [ok_cv_results['r2'], uk_cv_results['r2'], rk_cv_results['r2']],
})

print("\nCross-Validation Comparison:")
print(comparison.to_string(index=False))

# Plot comparison
fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=150)

axes[0].bar(comparison['Method'], comparison['RMSE'], color=['blue', 'orange', 'green'])
axes[0].set_ylabel('RMSE'); axes[0].set_title('Root Mean Square Error')
axes[0].grid(True, alpha=0.3, axis='y')

axes[1].bar(comparison['Method'], comparison['MAE'], color=['blue', 'orange', 'green'])
axes[1].set_ylabel('MAE'); axes[1].set_title('Mean Absolute Error')
axes[1].grid(True, alpha=0.3, axis='y')

axes[2].bar(comparison['Method'], comparison['R²'], color=['blue', 'orange', 'green'])
axes[2].set_ylabel('R²'); axes[2].set_title('R-Squared Score')
axes[2].set_ylim([0, 1])
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()